# StepStone Job Listings Scraper

This notebook contains code to scrape job listings from StepStone, extract relevant information, and store it in a structured format for further analysis.

## Import Required Libraries

First, let's import all the necessary libraries for web scraping and data handling.

In [ ]:
# Import libraries for requests, parsing HTML, data manipulation and utilities
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random
import json
from datetime import datetime
import re
import os
from typing import Dict, List, Optional, Union
from urllib.parse import urljoin
import logging

# Configure logging with color formatting
import colorama
from colorama import Fore, Style

colorama.init()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)


# Custom logger function with color coding
def log_message(level, message):
    if level == "ERROR":
        print(f"{Fore.RED}[ERROR] {message}{Style.RESET_ALL}")
    elif level == "WARNING":
        print(f"{Fore.YELLOW}[WARNING] {message}{Style.RESET_ALL}")
    elif level == "SUCCESS":
        print(f"{Fore.GREEN}[SUCCESS] {message}{Style.RESET_ALL}")
    else:
        print(f"{Fore.CYAN}[INFO] {message}{Style.RESET_ALL}")

## Define Scraper Configuration

Now, let's set up the base URL, headers, and parameters for our scraper. We'll define functions to construct proper URLs and request parameters.

In [ ]:
class StepStoneScraper:
    """
    A scraper for extracting job listings from StepStone.
    """

    def __init__(
        self,
        job_title: str = "data engineer",
        location: str = "berlin",
        radius: int = 30,
        page_limit: int = 3,
    ):
        """
        Initialize the StepStone scraper.

        Args:
            job_title: The job title to search for
            location: The location to search in
            radius: The radius around the location in kilometers
            page_limit: Maximum number of pages to scrape
        """
        self.base_url = "https://www.stepstone.de/jobs"
        self.search_url = "https://www.stepstone.de/5/ergebnisliste.html"
        self.headers = {
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3",
            "Accept-Language": "en-US,en;q=0.9",
            "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8",
            "Connection": "keep-alive",
            "Upgrade-Insecure-Requests": "1",
        }
        self.job_title = job_title.replace(" ", "-")
        self.location = location
        self.radius = radius
        self.page_limit = page_limit
        self.job_data = []

    def build_search_url(self, page: int = 0) -> str:
        """
        Build the search URL with the specified parameters.

        Args:
            page: The page number to request

        Returns:
            The complete search URL
        """
        params = {
            "what": self.job_title,
            "where": self.location,
            "radius": self.radius,
            "page": page,
        }

        # Convert parameters to URL format
        param_strings = []
        for key, value in params.items():
            if value:
                param_strings.append(f"{key}={value}")

        # Construct the URL
        url = f"{self.search_url}?{'&'.join(param_strings)}"
        log_message("INFO", f"Built URL: {url}")
        return url

## Extract Job Listings from HTML

Now we'll implement the functions to parse the HTML content and extract job details.

In [ ]:
    def extract_job_details(self, job_element) -> Dict:
        """
        Extract job details from a job listing element.
        
        Args:
            job_element: BeautifulSoup element containing job details
            
        Returns:
            Dictionary with extracted job information
        """
        try:
            # Extract job ID from data attribute or URL
            job_id = ""
            job_link_element = job_element.select_one("a[data-at='job-item-title']")
            if job_link_element:
                job_url = job_link_element.get('href', '')
                # Extract job ID from URL
                job_id_match = re.search(r'\/job-([a-zA-Z0-9-]+)\/', job_url)
                if job_id_match:
                    job_id = job_id_match.group(1)
                
            # Extract job title
            title_element = job_element.select_one("span[data-at='job-item-title']")
            title = title_element.text.strip() if title_element else "N/A"
            
            # Extract company name
            company_element = job_element.select_one("span[data-at='job-item-company-name']")
            company = company_element.text.strip() if company_element else "N/A"
            
            # Extract location
            location_element = job_element.select_one("span[data-at='job-item-location']")
            location = location_element.text.strip() if location_element else "N/A"
            
            # Extract company logo URL
            logo_element = job_element.select_one("img.at-company-logo")
            company_logo_url = logo_element.get('src', '') if logo_element else ""
            
            # Extract home office options
            homeoffice_element = job_element.select_one("li.sc-fznMAR span:contains('Home Office')")
            homeoffice_options = homeoffice_element.text.strip() if homeoffice_element else "N/A"
            
            # Extract salary if available
            salary_element = job_element.select_one("span.sc-fzqMAW")
            salary = salary_element.text.strip() if salary_element else "N/A"
            
            # Check if fast application is available
            fast_application = bool(job_element.select_one("span:contains('Schnelle Bewerbung')"))
            
            # Extract posting date if available
            date_element = job_element.select_one("time")
            posting_date = date_element.text.strip() if date_element else "N/A"
            
            # Extract job URL
            job_url = urljoin(self.base_url, job_link_element.get('href', '')) if job_link_element else ""
            
            return {
                "job_id": job_id,
                "title": title,
                "company": company,
                "location": location,
                "company_logo_url": company_logo_url,
                "homeoffice_options": homeoffice_options,
                "salary": salary,
                "fast_application": fast_application,
                "posting_date": posting_date,
                "job_url": job_url,
                "scrape_date": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
            }
        except Exception as e:
            log_message("ERROR", f"Error extracting job details: {str(e)}")
            return {}

## Handle Pagination

Let's implement the logic to navigate through multiple pages of job listings and collect data from each page.

In [ ]:
    def scrape_page(self, page: int = 0) -> List[Dict]:
        """
        Scrape a single page of job listings.
        
        Args:
            page: The page number to scrape
            
        Returns:
            List of dictionaries containing job details
        """
        url = self.build_search_url(page)
        
        try:
            log_message("INFO", f"Requesting page {page + 1}...")
            response = requests.get(url, headers=self.headers)
            response.raise_for_status()
            
            log_message("SUCCESS", f"Retrieved page {page + 1} with status code {response.status_code}")
            
            # Parse the HTML content
            soup = BeautifulSoup(response.content, "html.parser")
            
            # Find all job listing elements
            job_elements = soup.select("article.sc-dkPtRN")
            log_message("INFO", f"Found {len(job_elements)} job listings on page {page + 1}")
            
            # Extract job details from each element
            page_jobs = []
            for job_element in job_elements:
                job_details = self.extract_job_details(job_element)
                if job_details:
                    page_jobs.append(job_details)
            
            log_message("SUCCESS", f"Extracted {len(page_jobs)} job details from page {page + 1}")
            return page_jobs
            
        except requests.exceptions.RequestException as e:
            log_message("ERROR", f"Request error on page {page + 1}: {str(e)}")
            return []
        except Exception as e:
            log_message("ERROR", f"Error scraping page {page + 1}: {str(e)}")
            return []
            
    def scrape_all_pages(self) -> List[Dict]:
        """
        Scrape multiple pages of job listings up to the page limit.
        
        Returns:
            List of all scraped job details across pages
        """
        all_jobs = []
        
        for page in range(self.page_limit):
            log_message("INFO", f"Scraping page {page + 1} of {self.page_limit}...")
            page_jobs = self.scrape_page(page)
            all_jobs.extend(page_jobs)
            
            # Add a delay to avoid being blocked
            if page < self.page_limit - 1:
                delay = random.uniform(2.0, 5.0)
                log_message("INFO", f"Waiting for {delay:.2f} seconds before the next request...")
                time.sleep(delay)
        
        log_message("SUCCESS", f"Scraped a total of {len(all_jobs)} job listings across {self.page_limit} pages")
        self.job_data = all_jobs
        return all_jobs

## Collect and Store Job Data

Now let's implement methods to store the extracted job data in a structured format and save it to a CSV file.

In [ ]:
    def save_to_csv(self, filename: str = None) -> str:
        """
        Save the job data to a CSV file.
        
        Args:
            filename: Name of the file to save to (if None, a default name is generated)
            
        Returns:
            Path to the saved CSV file
        """
        if not self.job_data:
            log_message("WARNING", "No job data to save. Run scrape_all_pages() first.")
            return ""
            
        # Create a DataFrame from the job data
        df = pd.DataFrame(self.job_data)
        
        # Generate a filename if none is provided
        if filename is None:
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            job_title_slug = self.job_title.replace("-", "_")
            location_slug = self.location.replace(" ", "_")
            filename = f"stepstone_{job_title_slug}_{location_slug}_{timestamp}.csv"
        
        # Create the data directory if it doesn't exist
        os.makedirs("data", exist_ok=True)
        filepath = os.path.join("data", filename)
        
        # Save the DataFrame to a CSV file
        df.to_csv(filepath, index=False, encoding="utf-8")
        log_message("SUCCESS", f"Job data saved to {filepath}")
        
        # Display basic statistics
        log_message("INFO", f"Total jobs collected: {len(df)}")
        log_message("INFO", f"Companies found: {df['company'].nunique()}")
        log_message("INFO", f"Locations found: {df['location'].nunique()}")
        
        return filepath
        
    def get_dataframe(self) -> pd.DataFrame:
        """
        Get the job data as a pandas DataFrame.
        
        Returns:
            DataFrame containing the job data
        """
        return pd.DataFrame(self.job_data)

## Run the Scraper

Let's execute the scraper with the specified configuration and display the collected job data.

In [ ]:
# Initialize the scraper with search parameters
scraper = StepStoneScraper(
    job_title="data engineer", location="berlin", radius=30, page_limit=3
)

# Run the scraper to collect job data
jobs = scraper.scrape_all_pages()

# Convert to DataFrame and display the first few rows
df = scraper.get_dataframe()
display(df.head())

# Generate some basic insights
print("\n--- Job Market Insights ---")
print(f"Total jobs found: {len(df)}")
print(f"Number of unique companies: {df['company'].nunique()}")

# Top companies by job count
top_companies = df["company"].value_counts().head(5)
print("\nTop companies by job postings:")
display(top_companies)

# Distribution of locations
print("\nJob distribution by location:")
location_counts = df["location"].value_counts().head(5)
display(location_counts)

# Save the data to CSV
csv_path = scraper.save_to_csv()
print(f"\nData saved to: {csv_path}")

## Additional Analysis

Let's perform some additional analysis on the collected data to gain more insights.

In [ ]:
# Import visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

# Set the style for the plots
plt.style.use("ggplot")
sns.set(font_scale=1.2)

# Only proceed if we have data
if not df.empty:
    # Plot distribution of job postings by company (top 10)
    plt.figure(figsize=(12, 6))
    company_counts = df["company"].value_counts().head(10)
    company_counts.plot(kind="bar", color="skyblue")
    plt.title("Top 10 Companies by Number of Job Postings")
    plt.xlabel("Company")
    plt.ylabel("Number of Job Postings")
    plt.xticks(rotation=45, ha="right")
    plt.tight_layout()
    plt.show()

    # Analyze home office options if available
    if "homeoffice_options" in df.columns and df["homeoffice_options"].nunique() > 1:
        plt.figure(figsize=(10, 6))
        df["homeoffice_options"].fillna("Not Specified").value_counts().plot(
            kind="pie", autopct="%1.1f%%"
        )
        plt.title("Distribution of Home Office Options")
        plt.ylabel("")
        plt.tight_layout()
        plt.show()

    # Check for the presence of salary information
    salary_available = df["salary"].apply(lambda x: x != "N/A").sum()
    salary_percentage = (salary_available / len(df)) * 100
    print(
        f"Percentage of job postings with salary information: {salary_percentage:.2f}%"
    )

    # Analyze posting dates if available
    if "posting_date" in df.columns and df["posting_date"].nunique() > 1:
        df["posting_date_clean"] = pd.to_datetime(df["posting_date"], errors="coerce")
        recent_postings = df["posting_date_clean"].dt.date.value_counts().sort_index()

        plt.figure(figsize=(12, 5))
        recent_postings.plot(kind="line", marker="o")
        plt.title("Job Postings by Date")
        plt.xlabel("Posting Date")
        plt.ylabel("Number of Job Postings")
        plt.tight_layout()
        plt.show()

    # Check for fast application options
    if "fast_application" in df.columns:
        fast_app_count = df["fast_application"].sum()
        fast_app_percentage = (fast_app_count / len(df)) * 100
        print(
            f"Percentage of job postings with fast application option: {fast_app_percentage:.2f}%"
        )
else:
    print("No data available for analysis. Run the scraper first.")

## Next Steps and Improvements

Here are some potential improvements and next steps for this scraper:

1. Add more robust error handling and retry mechanisms
2. Implement proxy rotation to avoid IP bans
3. Extract more detailed job information (job description, requirements, etc.)
4. Implement a scheduling system to run the scraper periodically
5. Store data in a database instead of CSV files
6. Add sentiment analysis of job descriptions
7. Implement comparison with other job platforms
8. Create automated alerts for new relevant job postings